# AutoMix v3 — обучение на MTG-Jamendo (полный пайплайн)

Ноутбук делает всё сам, один запуск «Save & Run All (Commit)», ~6–8 часов:

1. **Скачивает метаданные** MTG-Jamendo (55 610 полных CC-треков) и отбирает
   ~15 000 треков **сбалансированно по жанрам** (электроника больше не «вся вселенная»).
2. **Качает аудио кусками** (tar-архивы low-версии, моно MP3) — распаковывает только
   нужные треки, считает фичи, удаляет архив. Диск не переполняется, на телефон не
   качается ни байта.
3. **Считает фичи ровно как приложение** (порт Kotlin-кода 1:1): мел-спектр 431×128
   (22050 Гц, FFT 2048, hop 512, dB от глобального максимума, (dB+80)/80) и aux16
   (BPM/200, 12 хром, RMS, центроид, onset).
4. **Псевдоразметка**: портированный алгоритм-учитель из приложения
   (BPMDetector + KeyDetector Крумханзл-Шмуклера + EnergyAnalyzer + правила
   SmartTransitionFinder/DJEffectsEngine) размечает ~30 000 пар: совместимость,
   тип перехода (6 классов), длительность кроссфейда, entry offset, точка старта.
5. **Обучает клон текущей архитектуры** (сиамский CNN-энкодер + 5 голов) на GPU.
6. **Экспортирует `automix_v2.tflite` (fp16)** с именами входов/выходов, которые
   ждёт `MLTransitionPredictor` — файл кладётся в приложение как drop-in замена.

### Перед запуском (панель Settings справа)
- **Internet → On** (нужна верификация аккаунта телефоном, если ещё не сделана)
- **Accelerator → GPU T4 x2 или P100**
- Потом: **Save Version → Save & Run All (Commit)**

### Результаты (вкладка Output у коммита)
- `automix_v2.tflite` — готовая модель для `app/src/main/assets/`
- `report.json`, `confusion_matrix.png`, `history.json` — метрики
- `dataset_stats.json` — что реально скачалось/разметилось

Если время сессии кончится раньше — ноутбук сам перестаёт качать по бюджету
(`DOWNLOAD_BUDGET_MIN`) и обучается на том, что успел собрать.

*Датасет MTG-Jamendo — только для некоммерческого исследовательского
использования; для коммерческого релиза модели, обученной на нём, нужно
разрешение Jamendo (hello@jamendo.com).*


In [ ]:
# ============================== КОНФИГ ==============================
import os, time

RUN_T0 = time.time()

CFG = dict(
    # --- отбор треков ---
    TARGET_TRACKS   = 15000,   # сколько треков хотим собрать
    MAX_PER_ARTIST  = 8,       # не больше N треков одного артиста (анти-перекос)
    MIN_DUR_S       = 90,      # отсекаем джинглы/огрызки
    MAX_DUR_S       = 600,     # и часовые DJ-сеты/эмбиент-полотна

    # --- бюджеты времени (минуты от старта ноутбука) ---
    DOWNLOAD_BUDGET_MIN = 270, # качаем/обрабатываем архивы не дольше этого
                               # (дальше — обучение на собранном)

    # --- пары и обучение ---
    TARGET_PAIRS    = 30000,
    VAL_FRAC        = 0.10,    # валидация — по ТРЕКАМ (без утечки пар)
    BATCH           = 64,
    EPOCHS          = 40,
    LR              = 3e-4,
    SEED            = 20260703,

    # --- пути ---
    WORK    = '/kaggle/working',
    TMP     = '/kaggle/tmp',
    FEAT    = '/kaggle/working/features',   # шарды фич (по одному на tar)

    # --- источники (MTG-Jamendo, зеркало Финляндия) ---
    MIRROR  = 'https://cdn.freesound.org/mtg-jamendo/raw_30s/audio-low/',
    META_TSV = ('https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/'
                'master/data/raw_30s_cleantags_50artists.tsv'),
    TARS_SHA = ('https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/'
                'master/data/download/raw_30s_audio-low_sha256_tars.txt'),
)

os.makedirs(CFG['TMP'], exist_ok=True)
os.makedirs(CFG['FEAT'], exist_ok=True)

def elapsed_min():
    return (time.time() - RUN_T0) / 60.0

print('Конфиг готов. Бюджет на скачивание: %d мин.' % CFG['DOWNLOAD_BUDGET_MIN'])


In [ ]:
# ============ ПОРТ ФИЧЕЙ ПРИЛОЖЕНИЯ (Kotlin -> numpy, 1:1) ============
# Источники: MelSpectrogram.kt, FeatureExtractor.kt, FFT.kt.
# Любое отклонение здесь = модель на телефоне видит "не те" данные,
# поэтому повторяем даже причуды (симметричное окно Ханна, паддинг
# кадров нулями в dB-домене => 0 dB => 1.0 после нормировки, и т.д.).
import numpy as np

SR, N_FFT, HOP, N_MELS = 22050, 2048, 512, 128
SEG_SEC       = 10
SEG_SAMPLES   = SEG_SEC * SR            # 220500
TARGET_FRAMES = 431                      # как в MelSpectrogram.TARGET_FRAMES

# Окно Ханна как в Kotlin: 0.5 - 0.5*cos(2*pi*i/(N-1)) — СИММЕТРИЧНОЕ
HANN = (0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(N_FFT) / (N_FFT - 1))).astype(np.float32)

def _hz_to_mel(hz):  return 2595.0 * np.log10(1.0 + hz / 700.0)
def _mel_to_hz(mel): return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def _build_mel_bank():
    """Порт MelSpectrogram.buildMelFilterBank: треугольники БЕЗ нормировки,
    границы бинов через floor((nFft+1)*hz/sr)."""
    n_bins = N_FFT // 2 + 1
    mel_pts = _hz_to_mel(0.0) + (_hz_to_mel(SR / 2.0) - _hz_to_mel(0.0)) * \
              np.arange(N_MELS + 2) / (N_MELS + 1)
    hz_pts  = _mel_to_hz(mel_pts)
    bin_pts = np.floor((N_FFT + 1) * hz_pts / SR).astype(int)
    bank = np.zeros((N_MELS, n_bins), dtype=np.float32)
    for m in range(1, N_MELS + 1):
        left, center, right = bin_pts[m - 1], bin_pts[m], bin_pts[m + 1]
        for k in range(left, center):
            if 0 <= k < n_bins and center != left:
                bank[m - 1, k] = (k - left) / (center - left)
        for k in range(center, right):
            if 0 <= k < n_bins and right != center:
                bank[m - 1, k] = (right - k) / (right - center)
    return bank

MEL_BANK_T = _build_mel_bank().T          # (1025, 128)

def _frames(x, window=None):
    """Нарезка на кадры (n,2048) с hop 512, как в Kotlin (без центрирования)."""
    n = (len(x) - N_FFT) // HOP + 1
    if n < 1:
        return np.zeros((0, N_FFT), dtype=np.float32)
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    fr = x[idx]
    return fr * window if window is not None else fr

def fit_segment(x, n=SEG_SAMPLES):
    """fitToExactSamples: обрезать или добить нулями до ровно n сэмплов."""
    x = np.asarray(x, dtype=np.float32)
    if len(x) >= n:
        return x[:n]
    out = np.zeros(n, dtype=np.float32)
    out[:len(x)] = x
    return out

def app_mel(seg):
    """MelSpectrogram.generate + FeatureExtractor.normalizeFrames.
    Вход: ровно 220500 сэмплов. Выход: (431,128) float32 в [0,1]."""
    fr   = _frames(seg, HANN)                          # (427, 2048)
    mag  = np.abs(np.fft.rfft(fr, n=N_FFT, axis=1))    # (427, 1025)
    mel  = np.maximum((mag * mag) @ MEL_BANK_T, 1e-10) # (427, 128)
    gmax = max(mel.max(), 1e-10)
    db   = np.maximum(10.0 * np.log10(mel / gmax), -80.0)
    # fitFramesToTarget: добивка до 431 кадрами НУЛЕЙ (= 0 dB, как в Kotlin)
    if db.shape[0] < TARGET_FRAMES:
        db = np.vstack([db, np.zeros((TARGET_FRAMES - db.shape[0], N_MELS), np.float32)])
    else:
        db = db[:TARGET_FRAMES]
    return np.clip((db + 80.0) / 80.0, 0.0, 1.0).astype(np.float32)

# --- отображение FFT-бин -> хрома-бин для aux-хромы (20..5000 Гц, БЕЗ окна) ---
def _chroma_map(fmin, fmax, trunc_before_mod):
    n_bins = N_FFT // 2 + 1
    M = np.zeros((n_bins, 12), dtype=np.float32)
    for k in range(1, n_bins):
        f = k * SR / N_FFT
        if f < fmin or f > fmax:
            continue
        midi = 12.0 * np.log2(f / 440.0) + 69.0
        if trunc_before_mod:
            b = (int(midi) % 12 + 12) % 12       # KeyDetector: toInt() ДО %12
        else:
            b = int((midi % 12.0 + 12.0) % 12.0)  # FeatureExtractor: %12 до toInt()
        M[k, b] = 1.0
    return M

CHROMA_AUX = _chroma_map(20.0, 5000.0, trunc_before_mod=False)   # (1025,12)

def app_aux16(seg):
    """FeatureExtractor.computeAuxFeatures: [bpm/200, chroma12, rms, centroid, onset].
    Всё на кадрах БЕЗ окна, как в Kotlin."""
    fr  = _frames(seg)                                  # без окна!
    mag = np.abs(np.fft.rfft(fr, n=N_FFT, axis=1))      # (427,1025)
    out = np.zeros(16, dtype=np.float32)

    # onset envelope = положительный спектральный поток (все бины)
    flux = np.maximum(mag[1:] - mag[:-1], 0.0).sum(axis=1) if len(mag) >= 2 \
           else np.zeros(0, np.float32)

    # BPM: автокорреляция envelope, лаги для 200..60 BPM
    bpm = 120.0
    if len(flux) >= 4:
        min_lag = int(60.0 / 200.0 * SR / HOP)          # 12
        max_lag = min(int(60.0 / 60.0 * SR / HOP), len(flux) - 1)
        best_lag, best_corr = min_lag, -1.0
        for lag in range(min_lag, max_lag + 1):
            v = flux[:len(flux) - lag]
            c = float((v * flux[lag:lag + len(v)]).mean()) if len(v) else -1.0
            if c > best_corr:
                best_corr, best_lag = c, lag
        bpm = float(np.clip(60.0 * SR / HOP / best_lag, 60.0, 200.0))
    out[0] = bpm / 200.0

    # chroma (энергия |X|^2, нормировка на frameCount*max)
    ch = ((mag * mag) @ CHROMA_AUX).sum(axis=0)
    if len(fr) > 0:
        ch = ch / (len(fr) * max(ch.max(), 1e-10))
    out[1:13] = ch

    out[13] = float(np.sqrt(np.mean(seg * seg))) if len(seg) else 0.0   # RMS

    # спектральный центроид: ОДИН кадр из середины, без окна, все бины (вкл. DC)
    mid = max((len(seg) - N_FFT) // 2, 0)
    frame = np.zeros(N_FFT, np.float32)
    avail = min(N_FFT, len(seg) - mid)
    frame[:avail] = seg[mid:mid + avail]
    m1 = np.abs(np.fft.rfft(frame, n=N_FFT))
    s = float(m1.sum())
    if s > 0:
        freqs = np.arange(len(m1)) * SR / N_FFT
        out[14] = float(np.clip((freqs * m1).sum() / s / (SR / 2.0), 0.0, 1.0))

    out[15] = float(np.clip(flux.mean() / 10.0, 0.0, 1.0)) if len(flux) else 0.0
    return out

# самопроверка на синусе
_t = np.arange(SEG_SAMPLES, dtype=np.float32) / SR
_test = (0.5 * np.sin(2 * np.pi * 440.0 * _t)).astype(np.float32)
_m = app_mel(_test); _a = app_aux16(_test)
assert _m.shape == (431, 128) and 0.0 <= _m.min() and _m.max() <= 1.0
assert _a.shape == (16,) and abs(_a[13] - 0.3536) < 0.01   # RMS синуса 0.5/sqrt(2)
print('DSP-порт готов: mel', _m.shape, 'aux16 ok, A4 хрома-пик =', int(np.argmax(_a[1:13])))


In [ ]:
# ============ ПОРТ АЛГОРИТМА-УЧИТЕЛЯ (псевдоразметка пар) ============
# Порт: BPMDetector.kt, KeyDetector.kt, EnergyAnalyzer.kt,
# SmartTransitionFinder.kt, DJEffectsEngine.kt.
# Учитель смотрит на 30-секундные хвост/голову (приближение: в приложении
# энергия считается по всему треку, но результат outro всё равно прижат
# к последним 20с, поэтому 30с хвоста достаточно).
import numpy as np

CHROMA_KEY = _chroma_map(65.0, 2000.0, trunc_before_mod=True)   # KeyDetector: A2..B6

MAJOR_PROFILE = np.array([6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88], np.float32)
MINOR_PROFILE = np.array([6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17], np.float32)

def _pearson(x, y):
    dx, dy = x - x.mean(), y - y.mean()
    d = np.sqrt((dx * dx).sum()) * np.sqrt((dy * dy).sum())
    return float((dx * dy).sum() / d) if d > 1e-4 else 0.0

def detect_key(x):
    """KeyDetector.detectKey: хрома с окном Ханна (65..2000 Гц) + Крумханзл-Шмуклер.
    Возвращает (key 0-11, is_minor, confidence) или None."""
    if len(x) < SR:
        return None
    fr = _frames(x, HANN)
    if len(fr) < 1:
        return None
    mag = np.abs(np.fft.rfft(fr, n=N_FFT, axis=1))
    ch = ((mag * mag) @ CHROMA_KEY).sum(axis=0) / len(fr)
    if ch.max() <= 0:
        return None
    ch = (ch / ch.max()).astype(np.float32)
    best = (-1e9, 0, False)
    for shift in range(12):
        rot_mj = np.array([MAJOR_PROFILE[(i - shift) % 12] for i in range(12)], np.float32)
        rot_mn = np.array([MINOR_PROFILE[(i - shift) % 12] for i in range(12)], np.float32)
        cmj, cmn = _pearson(ch, rot_mj), _pearson(ch, rot_mn)
        if cmj > best[0]: best = (cmj, shift, False)
        if cmn > best[0]: best = (cmn, shift, True)
    return (best[1], best[2], float(np.clip((best[0] + 1) / 2, 0, 1)))

def detect_bpm(x):
    """BPMDetector.detectBPM: 40-полосный спектральный поток (окно Ханна),
    автокорреляция 60..200 BPM, параболическое уточнение."""
    if len(x) < SR * 2:
        return None
    fr = _frames(x, HANN)
    if len(fr) < 2:
        return None
    mag = np.abs(np.fft.rfft(fr, n=N_FFT, axis=1))
    bpb = max(1, mag.shape[1] // 40)                      # binsPerBand = 25
    bands = np.stack([ (mag[:, b*bpb:min((b+1)*bpb, mag.shape[1])] ** 2).sum(axis=1)
                       for b in range(40) ], axis=1)
    flux = np.maximum(bands[1:] - bands[:-1], 0.0).sum(axis=1)
    if flux.max() > 0:
        flux = flux / flux.max()
    if len(flux) < 10:
        return None
    hop_rate = SR / HOP
    min_lag = int(round(hop_rate * 60.0 / 200.0))
    max_lag = int(round(hop_rate * 60.0 / 60.0))
    if max_lag >= len(flux):
        return None
    ac = np.zeros(max_lag + 1, np.float32)
    for lag in range(min_lag, max_lag + 1):
        v = flux[:len(flux) - lag]
        ac[lag] = float((v * flux[lag:lag + len(v)]).mean())
    best = int(np.argmax(ac[min_lag:max_lag + 1])) + min_lag
    lag_f = float(best)
    if min_lag < best < max_lag:
        a, b, c = ac[best - 1], ac[best], ac[best + 1]
        den = 2.0 * (2.0 * b - a - c)
        if den > 1e-3:
            lag_f = best + (a - c) / den
    bpm = hop_rate * 60.0 / lag_f
    return float(bpm) if 60.0 <= bpm <= 200.0 else None

def energy_curve(x):
    """EnergyAnalyzer: RMS по 100мс, сглаживание окном 5, нормировка на max."""
    spb = SR // 10
    n = len(x) // spb
    if n < 4:
        return np.array([0.5], np.float32)
    c = np.sqrt((x[:n * spb].reshape(n, spb) ** 2).mean(axis=1))
    sm = np.array([c[max(0, i - 2):min(n, i + 3)].mean() for i in range(n)], np.float32)
    m = sm.max()
    return sm / m if m > 0 else sm

def bpm_compat(b1, b2):
    """BPMDetector.bpmCompatibility (с учётом кратных 2x/0.5x)."""
    if b1 is None or b2 is None:
        return 0.5
    d = min(abs(b1 - b2), abs(b1 - b2 * 2), abs(b1 * 2 - b2))
    return 1.0 if d < 2 else 0.9 if d < 5 else 0.7 if d < 10 else 0.4 if d < 20 else 0.15

def key_compat(k1, k2):
    """KeyDetector.keyCompatibility (Camelot)."""
    if k1 is None or k2 is None:
        return 0.5
    (key1, m1, _), (key2, m2, _) = k1, k2
    st = (key2 - key1 + 12) % 12
    if st == 0 and m1 == m2: return 1.0
    if st == 0:              return 0.85
    if m1 and not m2 and st == 3: return 0.9
    if not m1 and m2 and st == 9: return 0.9
    if st in (7, 5):  return 0.75
    if st in (2, 10): return 0.55
    return 0.2

def energy_compat(e1, e2):
    d = abs(e1 - e2)
    return 1.0 if d < 0.1 else 0.85 if d < 0.2 else 0.6 if d < 0.35 else 0.4 if d < 0.5 else 0.2

def select_type(ea, eb, bc, kc):
    """DJEffectsEngine.selectTransitionType. 0..5."""
    if bc > 0.85 and kc > 0.6:   return 2   # BEAT_MATCH
    if ea > 0.7 and eb > 0.7:    return 1   # ENERGY_FADE
    if abs(ea - eb) > 0.3:       return 4   # FILTER_SWEEP
    if ea < 0.4 and eb < 0.4:    return 5   # ECHO_OUT
    if ea < 0.2 or eb < 0.2:     return 3   # HARD_CUT
    return 0                                # SMOOTH_FADE

def crossfade_ms(ttype, bpm_a, ea, eb):
    """DJEffectsEngine.calculateCrossfadeDuration."""
    base = {0: 8000, 1: 5000, 2: 12000, 3: 5000, 4: 10000, 5: 7000}[ttype]
    if ttype == 2 and bpm_a is not None:
        bar = int(60000.0 / bpm_a) * 4
        bars = 4 if bar * 4 < 16000 else 2
        return float(np.clip(bar * bars, 5000, 30000))
    avg = (ea + eb) / 2.0
    mult = 0.8 if avg > 0.7 else 1.3 if avg < 0.3 else 1.0
    return float(np.clip(base * mult, 5000, 30000))

def outro_start_ms(tail_curve, dur_ms, tail_ms):
    """EnergyAnalyzer.findOutroStart на кривой хвоста: 3 бина подряд ниже
    0.6*avg; результат в абсолютных мс, floor = dur-20с (как в приложении)."""
    thr = 0.6 * float(tail_curve.mean())
    drop = len(tail_curve) - 1
    for i in range(len(tail_curve) - 2):
        if tail_curve[i] < thr and tail_curve[i + 1] < thr and tail_curve[i + 2] < thr:
            drop = i
            break
    abs_ms = dur_ms - tail_ms + drop * 100.0
    return max(abs_ms, dur_ms - 20000.0)

def intro_end_ms(head_curve):
    """EnergyAnalyzer.findIntroEnd: где энергия достигает 0.5*avg; cap 10с."""
    thr = 0.5 * float(head_curve.mean())
    rise = 0
    for i in range(len(head_curve)):
        if head_curve[i] >= thr:
            rise = i
            break
    return min(rise * 100.0, 10000.0)

def teacher_pair(A, B):
    """Полная разметка пары по THUMB-полям треков (см. cell обработки).
    Возвращает целевые значения для 5 голов (нормированные как ждёт модель).

    BPM для СОВМЕСТИМОСТИ — из aux (та же шумная оценка, что видит модель
    на устройстве): v1-урок — учитель мерил точным детектором, модель видела
    шумный aux, корреляция головы compat была ~0.13 (нечему учиться).
    Для длительности beatmatch остаётся точный teacher-BPM."""
    bc = bpm_compat(A.get('bpm_at', A['bpm']), B.get('bpm_ah', B['bpm']))
    kc = key_compat(A['key'], B['key'])
    ec = energy_compat(A['avg_e'], B['avg_e'])
    compat = float(np.clip(bc * 0.35 + kc * 0.30 + ec * 0.35, 0, 1))
    tt = select_type(A['avg_e'], B['avg_e'], bc, kc)
    cf = crossfade_ms(tt, A['bpm'], A['avg_e'], B['avg_e'])
    # SmartTransitionFinder: start = min(outro, dur-crossfade), не раньше dur/2
    start_ms = min(A['outro_ms'], A['dur_ms'] - cf)
    start_ms = max(start_ms, A['dur_ms'] / 2.0)
    return dict(
        compat   = compat,
        duration = (cf - 5000.0) / 25000.0,          # 0..1 (5..30с)
        offset   = B['intro_ms'] / 10000.0,          # 0..1 (0..10с)
        ttype    = tt,
        start    = float(np.clip(start_ms / max(A['dur_ms'], 1.0), 0, 1)),
    )

print('Учитель готов (порт BPM/Key/Energy/Type/Duration/Start).')


In [ ]:
# ============ МЕТАДАННЫЕ + БАЛАНСИРОВКА ПО ЖАНРАМ ============
import csv, io, json, random, urllib.request
from collections import defaultdict

random.seed(CFG['SEED'])
np.random.seed(CFG['SEED'])

def fetch(url):
    with urllib.request.urlopen(url, timeout=120) as r:
        return r.read()

meta_raw = fetch(CFG['META_TSV']).decode('utf-8', 'replace')
rows = list(csv.reader(io.StringIO(meta_raw), delimiter='\t'))[1:]
print('Треков в метаданных:', len(rows))

# Жанровые группы (подстроки тегов genre---*). Порядок = приоритет присвоения:
# редкие группы разбираются первыми, чтобы их не съели electronic/pop.
GENRE_GROUPS = [
    ('hiphop',      {'hiphop', 'rap', 'rnb', 'triphop'}),
    ('jazz_funk',   {'jazz', 'blues', 'funk', 'soul', 'swing', 'bossanova'}),
    ('classical',   {'classical', 'orchestral', 'opera', 'symphonic', 'choir'}),
    ('folk_world',  {'folk', 'popfolk', 'world', 'country', 'latin', 'reggae', 'ska',
                     'celtic', 'ethno', 'oriental', 'african', 'flamenco', 'balkan'}),
    ('rock_metal',  {'rock', 'metal', 'punk', 'punkrock', 'hardrock', 'poprock',
                     'alternative', 'indie', 'grunge', 'hardcore', 'heavymetal'}),
    ('electronic',  {'electronic', 'techno', 'house', 'trance', 'dance', 'edm',
                     'drumnbass', 'dubstep', 'electropop', 'downtempo', 'idm',
                     'breakbeat', 'club', 'synthpop', 'eurodance', 'minimal',
                     'electronica', 'deephouse'}),
    ('ambient',     {'ambient', 'chillout', 'lounge', 'newage', 'atmospheric',
                     'relaxing', 'meditative', 'drone', 'soundscape'}),
    ('pop',         {'pop', 'instrumentalpop', 'chanson', 'disco', 'easylistening'}),
    ('soundtrack',  {'soundtrack', 'filmscore', 'cinematic', 'videogame', 'trailer',
                     'instrumental', 'experimental'}),
]

def track_group(tags):
    genres = {t.split('---', 1)[1].strip().lower() for t in tags if t.startswith('genre---')}
    for name, keys in GENRE_GROUPS:
        if genres & keys:
            return name
    return None   # без жанровых тегов — пропускаем (нужна балансировка)

# кандидаты: длительность ок + есть жанр
cands = defaultdict(list)   # group -> [(track_id, artist, path, dur_s)]
for r in rows:
    if len(r) < 6:
        continue
    tid, artist, path, dur = r[0], r[1], r[3], float(r[4])
    if not (CFG['MIN_DUR_S'] <= dur <= CFG['MAX_DUR_S']):
        continue
    g = track_group(r[5:])
    if g:
        cands[g].append((tid, artist, path, dur))

for g in cands:
    random.shuffle(cands[g])

# квоты: поровну на группу, дефицит перераспределяется
quota = {g: CFG['TARGET_TRACKS'] // len(GENRE_GROUPS) for g, _ in GENRE_GROUPS}
selected, per_artist = [], defaultdict(int)

def take(group, n):
    got = 0
    for tid, artist, path, dur in cands[group]:
        if got >= n:
            break
        if per_artist[artist] >= CFG['MAX_PER_ARTIST']:
            continue
        per_artist[artist] += 1
        selected.append(dict(tid=tid, path=path, dur_s=dur, group=group))
        got += 1
    return got

taken = {g: take(g, q) for g, q in quota.items()}
deficit = CFG['TARGET_TRACKS'] - len(selected)
for g, _ in sorted(GENRE_GROUPS, key=lambda kv: -len(cands[kv[0]])):   # добор из богатых групп
    if deficit <= 0:
        break
    extra = take(g, deficit)
    taken[g] += extra
    deficit -= extra

sel_by_folder = defaultdict(dict)    # '14' -> {'214': track}
for t in selected:
    folder, fname = t['path'].split('/', 1)
    sel_by_folder[folder][fname.split('.')[0]] = t

print('Отобрано %d треков:' % len(selected))
for g, n in sorted(taken.items(), key=lambda kv: -kv[1]):
    print('  %-12s %5d' % (g, n))
print('Папок (tar-архивов) затронуто:', len(sel_by_folder))

# контрольные суммы архивов (имена tar-файлов)
tars_txt = fetch(CFG['TARS_SHA']).decode()
TAR_NAMES = [ln.split()[1] for ln in tars_txt.strip().splitlines()]
print('Архивов в раздаче:', len(TAR_NAMES))


In [ ]:
# ============ СКАЧИВАНИЕ АРХИВОВ + ИЗВЛЕЧЕНИЕ ФИЧ ============
# Конвейер: фоновый поток качает следующий tar, пока пул из 8 воркеров
# обрабатывает текущий (ffmpeg-декод хвоста/головы -> мел/aux/учитель).
# После каждого tar — шард .npz и удаление архива. Перезапуск ноутбука
# пропускает уже готовые шарды.
import os, queue, subprocess, tarfile, threading
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor

TEACH_SEC = 30           # окно учителя (хвост/голова)
TEACH_SAMPLES = TEACH_SEC * SR

# ── Резюм: если к ноутбуку приложен Output прошлой версии (Add Input → свой
# ноутбук), подтягиваем уже готовые шарды фич — их скачивание пропустится. ──
import glob as _glob, shutil as _shutil
for _src in _glob.glob('/kaggle/input/*/features/feat_*.npz'):
    _dst = os.path.join(CFG['FEAT'], os.path.basename(_src))
    if not os.path.exists(_dst):
        _shutil.copy(_src, _dst)
_pre = len(_glob.glob(os.path.join(CFG['FEAT'], 'feat_*.npz')))
if _pre:
    print('Резюм: подхвачено готовых шардов из Input:', _pre)

def ffmpeg_decode(path, head, dur_s=TEACH_SEC):
    """Декод в моно float32 22050. head=True — первые dur_s, иначе последние."""
    cmd = ['ffmpeg', '-v', 'error']
    cmd += ['-ss', '0'] if head else ['-sseof', str(-dur_s)]
    cmd += ['-i', path, '-t', str(dur_s), '-ac', '1', '-ar', str(SR),
            '-f', 'f32le', 'pipe:1']
    try:
        out = subprocess.run(cmd, capture_output=True, timeout=60).stdout
        x = np.frombuffer(out, np.float32)
        return x if len(x) > SR else None       # меньше секунды = мусор
    except Exception:
        return None

def process_track(mp3_path, tinfo):
    """Все фичи одного трека. Возвращает dict или None."""
    tail = ffmpeg_decode(mp3_path, head=False)
    head = ffmpeg_decode(mp3_path, head=True)
    if tail is None or head is None:
        return None
    seg_tail = fit_segment(tail[-SEG_SAMPLES:])       # последние 10с — вход модели
    seg_head = fit_segment(head[:SEG_SAMPLES])        # первые 10с — вход модели
    tc, hc = energy_curve(tail), energy_curve(head)
    dur_ms = tinfo['dur_s'] * 1000.0
    return dict(
        tid      = tinfo['tid'],
        group    = tinfo['group'],
        mel_tail = app_mel(seg_tail).astype(np.float16),
        mel_head = app_mel(seg_head).astype(np.float16),
        aux_tail = app_aux16(seg_tail),
        aux_head = app_aux16(seg_head),
        bpm      = detect_bpm(tail) or detect_bpm(head),
        key      = detect_key(np.concatenate([head, tail])),
        avg_e    = float((np.concatenate([hc, tc])).mean()),
        outro_ms = outro_start_ms(tc, dur_ms, min(TEACH_SEC * 1000.0, dur_ms)),
        intro_ms = intro_end_ms(hc),
        dur_ms   = dur_ms,
    )

def download_tar(folder):
    name = 'raw_30s_audio-low-%s.tar' % folder
    dst = os.path.join(CFG['TMP'], name)
    if os.path.exists(dst):
        return dst
    with requests.get(CFG['MIRROR'] + name, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dst + '.part', 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    os.rename(dst + '.part', dst)
    return dst

folders = sorted(sel_by_folder.keys())
random.shuffle(folders)     # чтобы частичное скачивание осталось сбалансированным

# --- фоновая закачка со «складом» на 1 архив вперёд ---
dl_q = queue.Queue(maxsize=1)
def downloader():
    for folder in folders:
        shard = os.path.join(CFG['FEAT'], 'feat_%s.npz' % folder)
        if os.path.exists(shard):
            continue
        if elapsed_min() > CFG['DOWNLOAD_BUDGET_MIN']:
            break
        try:
            dl_q.put((folder, download_tar(folder)))
        except Exception as e:
            print('  ! скачивание %s: %s' % (folder, e))
    dl_q.put(None)
threading.Thread(target=downloader, daemon=True).start()

pool = ThreadPoolExecutor(max_workers=8)
done_tracks, failed = 0, 0

while True:
    item = dl_q.get()
    if item is None:
        break
    folder, tar_path = item
    wanted = sel_by_folder[folder]          # id -> tinfo
    jobs = []
    try:
        with tarfile.open(tar_path) as arch:
            for m in arch:
                if not m.isfile():
                    continue
                stem = os.path.basename(m.name).split('.')[0]
                if stem not in wanted:
                    continue
                mp3 = os.path.join(CFG['TMP'], 'trk_%s_%s.mp3' % (folder, stem))
                with open(mp3, 'wb') as f:
                    f.write(arch.extractfile(m).read())
                jobs.append((mp3, pool.submit(process_track, mp3, wanted[stem])))
    except Exception as e:
        print('  ! tar %s: %s' % (folder, e))

    shard = []
    for mp3, fut in jobs:
        try:
            r = fut.result(timeout=180)
            if r is not None:
                shard.append(r)
            else:
                failed += 1
        except Exception:
            failed += 1
        finally:
            try: os.remove(mp3)
            except OSError: pass
    try: os.remove(tar_path)
    except OSError: pass

    if shard:
        np.savez_compressed(
            os.path.join(CFG['FEAT'], 'feat_%s.npz' % folder),
            tids     = np.array([s['tid'] for s in shard]),
            groups   = np.array([s['group'] for s in shard]),
            mel_tail = np.stack([s['mel_tail'] for s in shard]),
            mel_head = np.stack([s['mel_head'] for s in shard]),
            aux_tail = np.stack([s['aux_tail'] for s in shard]),
            aux_head = np.stack([s['aux_head'] for s in shard]),
            bpm      = np.array([s['bpm'] if s['bpm'] else np.nan for s in shard], np.float32),
            key      = np.array([(s['key'][0], int(s['key'][1]), s['key'][2])
                                 if s['key'] else (-1, 0, 0.0) for s in shard], np.float32),
            avg_e    = np.array([s['avg_e'] for s in shard], np.float32),
            outro_ms = np.array([s['outro_ms'] for s in shard], np.float32),
            intro_ms = np.array([s['intro_ms'] for s in shard], np.float32),
            dur_ms   = np.array([s['dur_ms'] for s in shard], np.float32),
        )
        done_tracks += len(shard)
    print('tar %s: +%d треков (всего %d, битых %d, %.0f мин)' %
          (folder, len(shard), done_tracks, failed, elapsed_min()))

pool.shutdown(wait=True)
print('Сбор фич завершён: %d треков, %d не декодировалось, %.0f мин.' %
      (done_tracks, failed, elapsed_min()))


In [ ]:
# ============ СБОРКА ДАТАСЕТА ПАР + ПСЕВДОРАЗМЕТКА ============
import glob, json
import numpy as np

shards = sorted(glob.glob(os.path.join(CFG['FEAT'], 'feat_*.npz')))
assert shards, 'Нет ни одного шарда фич — скачивание не удалось.'

MT, MH, AT, AH, TRACKS = [], [], [], [], []
for sp in shards:
    z = np.load(sp, allow_pickle=False)
    MT.append(z['mel_tail']); MH.append(z['mel_head'])
    AT.append(z['aux_tail']); AH.append(z['aux_head'])
    for i in range(len(z['tids'])):
        k = z['key'][i]
        TRACKS.append(dict(
            group=str(z['groups'][i]),
            bpm=None if np.isnan(z['bpm'][i]) else float(z['bpm'][i]),
            # BPM «глазами модели» (aux[0]*200) — для разметки compat
            bpm_at=float(z['aux_tail'][i][0] * 200.0),
            bpm_ah=float(z['aux_head'][i][0] * 200.0),
            key=None if k[0] < 0 else (int(k[0]), bool(k[1]), float(k[2])),
            avg_e=float(z['avg_e'][i]), outro_ms=float(z['outro_ms'][i]),
            intro_ms=float(z['intro_ms'][i]), dur_ms=float(z['dur_ms'][i]),
        ))
MEL_TAIL = np.concatenate(MT); MEL_HEAD = np.concatenate(MH)
AUX_TAIL = np.concatenate(AT).astype(np.float32)
AUX_HEAD = np.concatenate(AH).astype(np.float32)
del MT, MH, AT, AH
N = len(TRACKS)
print('Треков в датасете: %d, мелы: %s + %s (fp16)' % (N, MEL_TAIL.shape, MEL_HEAD.shape))

# --- train/val по ТРЕКАМ (никаких общих треков между сплитами) ---
rng = np.random.default_rng(CFG['SEED'])
perm = rng.permutation(N)
n_val = max(int(N * CFG['VAL_FRAC']), 50)
val_ids, train_ids = set(perm[:n_val].tolist()), perm[n_val:].tolist()

by_group_train = defaultdict(list)
for i in train_ids:
    by_group_train[TRACKS[i]['group']].append(i)

def sample_pairs(track_pool, by_group, n_pairs):
    """65% пар внутри жанровой группы, 35% — между группами."""
    pairs, pool = [], list(track_pool)
    groups = [g for g in by_group if len(by_group[g]) >= 2]
    for _ in range(n_pairs):
        if groups and rng.random() < 0.65:
            g = groups[int(rng.integers(len(groups)))]
            a, b = rng.choice(by_group[g], 2, replace=False)
        else:
            a, b = rng.choice(pool, 2, replace=False)
        pairs.append((int(a), int(b)))
    return pairs

n_train_pairs = min(CFG['TARGET_PAIRS'], len(train_ids) * 3)
train_pairs = sample_pairs(train_ids, by_group_train, n_train_pairs)

by_group_val = defaultdict(list)
for i in val_ids:
    by_group_val[TRACKS[i]['group']].append(i)
val_pairs = sample_pairs(list(val_ids), by_group_val, max(n_train_pairs // 9, 200))

def label_pairs(pairs):
    y = dict(compat=[], duration=[], offset=[], ttype=[], start=[])
    for a, b in pairs:
        t = teacher_pair(TRACKS[a], TRACKS[b])
        for k in y:
            y[k].append(t[k])
    return {k: np.array(v, np.float32) for k, v in y.items()}

Y_TRAIN, Y_VAL = label_pairs(train_pairs), label_pairs(val_pairs)

hist = np.bincount(Y_TRAIN['ttype'].astype(int), minlength=6)
names = ['SMOOTH', 'ENERGY', 'BEATMATCH', 'HARDCUT', 'FILTER', 'ECHO']
print('Пар: train=%d val=%d' % (len(train_pairs), len(val_pairs)))
print('Типы переходов (train):', {names[i]: int(hist[i]) for i in range(6)})
print('compat: mean=%.3f std=%.3f' % (Y_TRAIN['compat'].mean(), Y_TRAIN['compat'].std()))

# Веса классов для головы типа: КОРЕНЬ из обратной частоты. v1-урок:
# линейные веса пережали — SMOOTH (мажоритарный) просел до recall 0.32,
# модель рассыпала плавные переходы в спецэффекты. sqrt мягче: SMOOTH
# остаётся уверенным, миноритарные всё ещё подтянуты.
cw = (hist.sum() / np.maximum(hist, 1) / 6.0) ** 0.5
TYPE_W_TRAIN = cw[Y_TRAIN['ttype'].astype(int)].astype(np.float32)
print('Веса классов (sqrt):', np.round(cw, 2))

json.dump({'tracks': N, 'train_pairs': len(train_pairs), 'val_pairs': len(val_pairs),
           'type_hist': hist.tolist(),
           'groups': {g: len(v) for g, v in by_group_train.items()}},
          open(os.path.join(CFG['WORK'], 'dataset_stats.json'), 'w'), indent=1)


In [ ]:
# ============ МОДЕЛЬ (клон архитектуры приложения) + ОБУЧЕНИЕ ============
import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(CFG['SEED'])

def make_mel_encoder():
    """Сиамский энкодер как в automix_v2.tflite: 4x[Conv3x3-BN-ReLU(+pool)] -> GAP
    -> Dense128 -> Dense64. Один экземпляр = общие веса для mel_a и mel_b."""
    inp = layers.Input((TARGET_FRAMES, N_MELS, 1))
    x = inp
    for filters, pool in [(32, True), (64, True), (128, True), (128, False)]:
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        if pool:
            x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64)(x)
    return Model(inp, x, name='mel_encoder')

def build_model():
    mel_a = layers.Input((TARGET_FRAMES, N_MELS, 1), name='mel_a')
    mel_b = layers.Input((TARGET_FRAMES, N_MELS, 1), name='mel_b')
    aux   = layers.Input((32,), name='aux')

    enc = make_mel_encoder()
    ea, eb = enc(mel_a), enc(mel_b)

    ax = layers.Dense(64, activation='relu')(aux)
    ax = layers.BatchNormalization()(ax)
    ax = layers.Dense(32, activation='relu')(ax)
    ax = layers.BatchNormalization()(ax)

    x = layers.Concatenate()([ea, eb, ax])              # 64+64+32 = 160
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)                          # (нет в оригинале; инференс не меняет)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)

    def head(units_out, act, name):
        h = layers.Dense(32, activation='relu')(x)
        return layers.Dense(units_out, activation=act, name=name)(h)

    # Keras 3: СЛОВАРИ входов/выходов — y/sample_weight из tf.data обязаны
    # совпадать по структуре с выходами модели (список ловил KeyError: 0).
    outs = {'y_compat':   head(1, 'sigmoid', 'y_compat'),
            'y_duration': head(1, 'sigmoid', 'y_duration'),
            'y_offset':   head(1, 'sigmoid', 'y_offset'),
            'y_type':     head(6, 'softmax', 'y_type'),
            'y_start':    head(1, 'sigmoid', 'y_start')}
    return Model({'mel_a': mel_a, 'mel_b': mel_b, 'aux': aux}, outs, name='AutoMix_v3')

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(CFG['LR']),
    loss={'y_compat': 'mse', 'y_duration': 'mse', 'y_offset': 'mse',
          'y_type': 'sparse_categorical_crossentropy', 'y_start': 'mse'},
    loss_weights={'y_compat': 2.0, 'y_duration': 1.0, 'y_offset': 1.0,
                  'y_type': 1.0, 'y_start': 1.0},
)
print(model.summary(line_length=100))

def make_ds(pairs, Y, type_w=None, shuffle=True):
    pa = np.array([p[0] for p in pairs]); pb = np.array([p[1] for p in pairs])
    idx = np.arange(len(pairs))
    def gen():
        order = np.random.permutation(idx) if shuffle else idx
        for i in order:
            a, b = pa[i], pb[i]
            xs = {'mel_a': MEL_TAIL[a].astype(np.float32)[..., None],
                  'mel_b': MEL_HEAD[b].astype(np.float32)[..., None],
                  'aux':   np.concatenate([AUX_TAIL[a], AUX_HEAD[b]])}
            ys = {'y_compat':   Y['compat'][i:i+1],
                  'y_duration': Y['duration'][i:i+1],
                  'y_offset':   Y['offset'][i:i+1],
                  'y_type':     Y['ttype'][i:i+1],
                  'y_start':    Y['start'][i:i+1]}
            one = np.ones(1, np.float32)
            ws = {'y_compat': one, 'y_duration': one, 'y_offset': one,
                  'y_start': one,
                  'y_type': (type_w[i:i+1] if type_w is not None else one)}
            yield xs, ys, ws
    sig = (
        {'mel_a': tf.TensorSpec((TARGET_FRAMES, N_MELS, 1), tf.float32),
         'mel_b': tf.TensorSpec((TARGET_FRAMES, N_MELS, 1), tf.float32),
         'aux':   tf.TensorSpec((32,), tf.float32)},
        {k: tf.TensorSpec((1,), tf.float32) for k in
         ['y_compat', 'y_duration', 'y_offset', 'y_type', 'y_start']},
        {k: tf.TensorSpec((1,), tf.float32) for k in
         ['y_compat', 'y_duration', 'y_offset', 'y_type', 'y_start']},
    )
    return (tf.data.Dataset.from_generator(gen, output_signature=sig)
            .batch(CFG['BATCH']).prefetch(4))

ds_train = make_ds(train_pairs, Y_TRAIN, TYPE_W_TRAIN, shuffle=True)
ds_val   = make_ds(val_pairs, Y_VAL, shuffle=False)

hist_obj = model.fit(
    ds_train, validation_data=ds_val, epochs=CFG['EPOCHS'],
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True,
                                         monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5,
                                             monitor='val_loss', verbose=1),
    ],
    verbose=2,
)
json.dump({k: [float(x) for x in v] for k, v in hist_obj.history.items()},
          open(os.path.join(CFG['WORK'], 'history.json'), 'w'))
model.save(os.path.join(CFG['WORK'], 'automix_v3.keras'))
print('Обучение завершено, %.0f мин от старта.' % elapsed_min())


In [ ]:
# ============ ОЦЕНКА: метрики по головам + матрица ошибок ============
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pred = model.predict(ds_val, verbose=0)   # dict-выходы (Keras 3)
p_compat = np.asarray(pred['y_compat']);  p_dur   = np.asarray(pred['y_duration'])
p_off    = np.asarray(pred['y_offset']);  p_type  = np.asarray(pred['y_type'])
p_start  = np.asarray(pred['y_start'])

def mae(a, b): return float(np.mean(np.abs(a.ravel() - b.ravel())))

report = {
    'val_pairs':        len(val_pairs),
    'compat_mae':       mae(p_compat, Y_VAL['compat']),
    'compat_corr':      float(np.corrcoef(p_compat.ravel(), Y_VAL['compat'])[0, 1]),
    'duration_mae_ms':  mae(p_dur, Y_VAL['duration']) * 25000.0,
    'offset_mae_ms':    mae(p_off, Y_VAL['offset']) * 10000.0,
    'start_mae_frac':   mae(p_start, Y_VAL['start']),
    'type_accuracy':    float((p_type.argmax(1) == Y_VAL['ttype'].astype(int)).mean()),
}

cm = np.zeros((6, 6), int)
for t, p in zip(Y_VAL['ttype'].astype(int), p_type.argmax(1)):
    cm[t, p] += 1
report['type_recall_per_class'] = [
    float(cm[i, i] / cm[i].sum()) if cm[i].sum() else None for i in range(6)]

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(names, rotation=45, ha='right'); ax.set_yticklabels(names)
ax.set_xlabel('модель'); ax.set_ylabel('учитель')
for i in range(6):
    for j in range(6):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(CFG['WORK'], 'confusion_matrix.png'), dpi=120)

json.dump(report, open(os.path.join(CFG['WORK'], 'report.json'), 'w'), indent=1)
print(json.dumps(report, indent=1))
print('\nВАЖНО смотреть type_recall_per_class: если какой-то класс ~0 — '
      'модель его игнорирует (мало примеров у учителя).')


In [ ]:
# ============ ЭКСПОРТ TFLITE FP16 + ПРОВЕРКА КОНТРАКТА ============
# Контракт с MLTransitionPredictor.kt:
#  - имена входов содержат "mel_a" / "mel_b" / "aux";
#  - выходы: приложение сначала матчит ПОДСТРОКИ имён, затем суффикс ":N"
#    (N = порядок голов Keras: 0=compat 1=duration 2=offset 3=type 4=start),
#    плюс железный гвард "голова с 6 элементами = transition_type".
#  Ключи словаря ниже подобраны так, что АЛФАВИТНЫЙ порядок (= порядок
#  выходов в TFLite) совпадает с семантикой суффиксов И содержит нужные
#  подстроки — контракт выполняется всеми тремя путями сразу.
import tensorflow as tf

class Export(tf.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    @tf.function(input_signature=[
        tf.TensorSpec([1, TARGET_FRAMES, N_MELS, 1], tf.float32, name='mel_a'),
        tf.TensorSpec([1, TARGET_FRAMES, N_MELS, 1], tf.float32, name='mel_b'),
        tf.TensorSpec([1, 32], tf.float32, name='aux'),
    ])
    def serve(self, mel_a, mel_b, aux):
        out = self.m({'mel_a': mel_a, 'mel_b': mel_b, 'aux': aux}, training=False)
        return {'a_compatibility':      out['y_compat'],     # :0
                'b_crossfade_duration': out['y_duration'],   # :1
                'c_entry_offset':       out['y_offset'],     # :2
                'd_transition_type':    out['y_type'],       # :3  (softmax [1,6])
                'e_transition_start':   out['y_start']}      # :4

exp = Export(model)
cf = exp.serve.get_concrete_function()
conv = tf.lite.TFLiteConverter.from_concrete_functions([cf], exp)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]      # fp16 как у текущей модели
tfl = conv.convert()

out_path = os.path.join(CFG['WORK'], 'automix_v2.tflite')
open(out_path, 'wb').write(tfl)
print('TFLite: %.1f КБ' % (len(tfl) / 1024))

# --- проверка интерпретатором: имена, формы, живость голов ---
it = tf.lite.Interpreter(model_content=tfl)
it.allocate_tensors()
ins, outs = it.get_input_details(), it.get_output_details()
print('\nВходы:');  [print(' ', d['name'], d['shape']) for d in ins]
print('Выходы:'); [print(' ', d['name'], d['shape']) for d in outs]

low = [d['name'].lower() for d in ins]
assert any('mel_a' in n for n in low) and any('mel_b' in n for n in low) \
   and any('aux' in n for n in low), 'Имена входов сломаны!'
assert sum(int(np.prod(d['shape'])) == 6 for d in outs) == 1, 'Нет головы [1,6]!'

# живость: 8 разных реальных пар из val -> выходы обязаны отличаться
rng2 = np.random.default_rng(1)
picks = rng2.choice(len(val_pairs), size=min(8, len(val_pairs)), replace=False)
results = []
for pi in picks:
    a, b = val_pairs[pi]
    feed = {'mel_a': MEL_TAIL[a].astype(np.float32)[None, ..., None],
            'mel_b': MEL_HEAD[b].astype(np.float32)[None, ..., None],
            'aux':   np.concatenate([AUX_TAIL[a], AUX_HEAD[b]])[None]}
    for d in ins:
        key = [k for k in feed if k in d['name'].lower()][0]
        it.set_tensor(d['index'], feed[key])
    it.invoke()
    results.append(np.concatenate([it.get_tensor(d['index']).ravel() for d in outs]))
results = np.stack(results)
spread = results.std(axis=0)
print('\nРазброс выходов по 8 парам (все > 0 => головы живые):')
print(np.round(spread, 4))
assert spread.max() > 1e-3, 'Выходы не зависят от входа — модель вырождена!'

print('\nГОТОВО за %.0f мин. Файл: %s -> положить в app/src/main/assets/' %
      (elapsed_min(), out_path))
